# AlphaTrain Iteration 1: Self-Play Training (RunPod)

Train the ResNet on 277K self-play states (493 games, 800 sims MCTS).
Policy learns MCTS visit distributions (KL-divergence). Value learns TD returns (MSE).

**Upload to `/workspace/`:**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_iter1.pt` — training data (8.9 GB)
3. `alphatrain_td_best.pt` — base model to fine-tune

**Download results:** `best.pt` and `latest.pt` from `/workspace/checkpoints/selfplay_iter1/`

In [ ]:
import os
os.chdir('/workspace')
!tar xzf colorlines_selfplay_train.tar.gz

# Move data files into place
os.makedirs('alphatrain/data', exist_ok=True)
for f in ['selfplay_iter1.pt', 'alphatrain_td_best.pt']:
    if os.path.exists(f'/workspace/{f}') and not os.path.exists(f'alphatrain/data/{f}'):
        os.rename(f'/workspace/{f}', f'alphatrain/data/{f}')
    print(f'{f}: {os.path.getsize(f"alphatrain/data/{f}")/1e6:.0f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Iteration 1: self-play training
# Policy: KL-divergence with MCTS visit distributions
# Value: MSE on scalar TD returns (sigmoid * 500)
# Warm start from epoch 6 heuristic model
!python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay_iter1.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 1.0 \
    --resume alphatrain/data/alphatrain_td_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 3e-4 --warmup-epochs 2 \
    --save-dir /workspace/checkpoints/selfplay_iter1

In [ ]:
# Evaluate: standalone policy score (baseline was 314)
!python -m alphatrain.evaluate --player policy \
    --model /workspace/checkpoints/selfplay_iter1/best.pt \
    --games 50 --seed 42

In [ ]:
!ls -lh /workspace/checkpoints/selfplay_iter1/